In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import IntSlider, FloatSlider, HTML, HBox, Layout
from IPython.display import display

# ============================================================
# FREQUENCY-SAMPLING IMPLEMENTATION
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12.5,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.fs-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.fs-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.fs-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.48;
    margin-bottom:7px;
}

.fs-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:6px;
    font-size:13.5px;
    line-height:1.45;
}

.fs-result{
    background:#fff8e6;
    border:1px solid #d8b451;
}

.fs-title{
    color:#0d47a1;
    font-weight:bold;
    font-size:14.5px;
    margin-bottom:5px;
}

.fs-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:6px 0;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="fs-root">

<div class="fs-header">
Frequency-Sampling Implementation — Parallel Resonators and Comb Section
</div>

<div class="fs-doc">

For α = 0, the frequency-sampling realization can be interpreted as the
cascade of two systems:

<div class="fs-equation">
<b>H(z) = H₁(z) H₂(z)</b>
</div>

with

<div class="fs-equation">
<b>
H₁(z) = (1 − z<sup>−M</sup>)/M
</b>
</div>

and

<div class="fs-equation">
<b>
H₂(z) =
Σ<sub>k=0</sub><sup>M−1</sup>
H[k] / (1 − e<sup>j2πk/M</sup>z<sup>−1</sup>).
</b>
</div>

Thus, H₂(z) is implemented as a <b>parallel resonator bank</b>. The outputs
of the active resonators are added and the resulting signal is then passed
through the common <b>comb section H₁(z)</b>.

<div class="fs-equation">
<b>
parallel resonator bank → summation → comb section
</b>
</div>

Each nonzero frequency sample H[k] activates one resonator branch.
In this notebook only a conjugate pair

<div class="fs-equation">
<b>
H[k₀] = H[M−k₀] = A
</b>
</div>

is nonzero. This produces a real FIR filter while making the structure
particularly easy to visualize.

The <b>k₀ slider</b> moves the active frequency pair around the DFT grid,
while the <b>A slider</b> changes its amplitude.

The direct FIR implementation is obtained independently from the IDFT of
the frequency samples and is used only as a numerical reference.

</div>

</div>
"""))

# ============================================================
# PARAMETERS
# ============================================================

M = 16
N = 96

n = np.arange(N)

# ============================================================
# CONTROLS
# ============================================================

k_slider = IntSlider(value=3,min=1,max=M//2-1,step=1,description='k₀:',continuous_update=True,style={'description_width':'30px'},layout=Layout(width='260px'))

amplitude_slider = FloatSlider(value=1.0,min=0.20,max=1.00,step=0.05,description='A:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='280px'))

controls = HBox([k_slider,amplitude_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='6px 9px',margin='0 0 3px 0'))

# ============================================================
# FIXED TEST INPUT
# ============================================================

x = np.zeros(N)

x[0] = 1.0

x += 0.30*np.sin(2*np.pi*2*n/M)
x += 0.25*np.sin(2*np.pi*3*n/M)
x += 0.20*np.sin(2*np.pi*5*n/M)
x += 0.15*np.sin(2*np.pi*7*n/M)

# ============================================================
# FREQUENCY-SAMPLE VECTOR
# ============================================================

def create_frequency_samples(k0,A):

    Hk = np.zeros(M,dtype=complex)

    Hk[k0] = A
    Hk[M-k0] = A

    return Hk

# ============================================================
# DIRECT FIR IMPLEMENTATION FROM IDFT
# ============================================================

def direct_fir_output(x,Hk):

    h = np.real_if_close(np.fft.ifft(Hk)).real

    y = signal.lfilter(h,[1.0],x)

    return h,y

# ============================================================
# FREQUENCY-SAMPLING IMPLEMENTATION
# ============================================================

def frequency_sampling_output(x,k0,A):

    omega1 = 2*np.pi*k0/M
    omega2 = 2*np.pi*(M-k0)/M

    pole1 = np.exp(1j*omega1)
    pole2 = np.exp(1j*omega2)

    v1 = np.zeros(len(x),dtype=complex)
    v2 = np.zeros(len(x),dtype=complex)

    for m in range(len(x)):

        previous1 = v1[m-1] if m >= 1 else 0.0
        previous2 = v2[m-1] if m >= 1 else 0.0

        v1[m] = x[m]+pole1*previous1
        v2[m] = x[m]+pole2*previous2

    resonator_sum = A*v1+A*v2

    y = np.zeros(len(x),dtype=complex)

    for m in range(len(x)):

        delayed = resonator_sum[m-M] if m >= M else 0.0

        y[m] = (resonator_sum[m]-delayed)/M

    return v1,v2,resonator_sum,np.real_if_close(y).real

# ============================================================
# FREQUENCY RESPONSE FROM IMPULSE RESPONSE
# ============================================================

def calculate_response(h):

    omega,H = signal.freqz(h,[1.0],worN=1024)

    return omega,H

# ============================================================
# PRECOMPUTE FIXED AXIS LIMITS
# ============================================================

all_internal = []
all_outputs = []
all_differences = []

for k0 in range(k_slider.min,k_slider.max+1):

    for A in [amplitude_slider.min,amplitude_slider.max]:

        Hk_test = create_frequency_samples(k0,A)

        h_test,y_direct_test = direct_fir_output(x,Hk_test)

        v1_test,v2_test,sum_test,y_fs_test = frequency_sampling_output(x,k0,A)

        all_internal.extend(np.real(A*v1_test))
        all_internal.extend(np.real(A*v2_test))
        all_internal.extend(np.real(sum_test))

        all_outputs.extend(y_direct_test)
        all_outputs.extend(y_fs_test)

        all_differences.extend(y_direct_test-y_fs_test)

internal_abs = max(np.max(np.abs(all_internal)),1.0)

output_abs = max(np.max(np.abs(all_outputs)),1.0)

difference_abs = max(np.max(np.abs(all_differences)),1e-14)

INTERNAL_LIMIT = 1.10*internal_abs
OUTPUT_LIMIT = 1.10*output_abs
DIFFERENCE_LIMIT = 1.20*difference_abs

# ============================================================
# INITIAL VALUES
# ============================================================

k0 = k_slider.value
A = amplitude_slider.value

Hk = create_frequency_samples(k0,A)

h,y_direct = direct_fir_output(x,Hk)

v1,v2,resonator_sum,y_fs = frequency_sampling_output(x,k0,A)

difference = y_direct-y_fs

omega,H_response = calculate_response(h)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML(layout=Layout(width=CONTENT_WIDTH))

# ============================================================
# FIGURE 1 — FREQUENCY-SAMPLING STRUCTURE
# ============================================================

fig1,ax_structure = plt.subplots(figsize=(9.0,3.0))

fig1.canvas.toolbar_visible = False
fig1.canvas.header_visible = False
fig1.canvas.footer_visible = False

ax_structure.set_xlim(0,10)

ax_structure.set_ylim(0,5)

ax_structure.axis('off')

ax_structure.set_title('Frequency-Sampling Structure')

# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------

ax_structure.text(0.35,2.70,r'$x[n]$',fontsize=11,fontweight='bold')

ax_structure.annotate('',xy=(1.35,2.70),xytext=(0.65,2.70),arrowprops={'arrowstyle':'->','linewidth':1.4})

# ------------------------------------------------------------
# FILTER-BANK INPUT BUS
# ------------------------------------------------------------

ax_structure.plot([1.35,1.35],[1.45,3.95],linewidth=1.3)

ax_structure.text(3.75,4.55,'Parallel resonator bank',ha='center',fontsize=10.5,fontweight='bold')

# ------------------------------------------------------------
# UPPER RESONATOR
# ------------------------------------------------------------

ax_structure.annotate('',xy=(2.25,3.75),xytext=(1.35,3.75),arrowprops={'arrowstyle':'->','linewidth':1.3})

upper_box = plt.Rectangle((2.25,3.35),3.25,0.80,fill=False,linewidth=1.3)

ax_structure.add_patch(upper_box)

upper_text = ax_structure.text(3.875,3.75,'',ha='center',va='center',fontsize=10.5)

# ------------------------------------------------------------
# LOWER RESONATOR
# ------------------------------------------------------------

ax_structure.annotate('',xy=(2.25,1.65),xytext=(1.35,1.65),arrowprops={'arrowstyle':'->','linewidth':1.3})

lower_box = plt.Rectangle((2.25,1.25),3.25,0.80,fill=False,linewidth=1.3)

ax_structure.add_patch(lower_box)

lower_text = ax_structure.text(3.875,1.65,'',ha='center',va='center',fontsize=10.5)

# ------------------------------------------------------------
# SUMMING NODE
# ------------------------------------------------------------

sum_x = 6.40
sum_y = 2.70
sum_radius = 0.21

ax_structure.annotate('',xy=(sum_x-sum_radius,2.82),xytext=(5.50,3.75),arrowprops={'arrowstyle':'->','linewidth':1.3})

ax_structure.annotate('',xy=(sum_x-sum_radius,2.58),xytext=(5.50,1.65),arrowprops={'arrowstyle':'->','linewidth':1.3})

sum_circle = plt.Circle((sum_x,sum_y),sum_radius,fill=False,linewidth=1.4)

ax_structure.add_patch(sum_circle)

ax_structure.text(sum_x,sum_y,r'$\Sigma$',ha='center',va='center',fontsize=12,fontweight='bold')

# ------------------------------------------------------------
# COMB SECTION
# ------------------------------------------------------------

ax_structure.annotate('',xy=(7.05,2.70),xytext=(sum_x+sum_radius,2.70),arrowprops={'arrowstyle':'->','linewidth':1.4})

comb_box = plt.Rectangle((7.05,2.27),1.70,0.86,fill=False,linewidth=1.3)

ax_structure.add_patch(comb_box)

ax_structure.text(7.90,2.70,r'$(1-z^{-M})/M$',ha='center',va='center',fontsize=10.5)

ax_structure.text(7.90,3.55,'Comb section',ha='center',fontsize=10.5,fontweight='bold')

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

ax_structure.annotate('',xy=(9.55,2.70),xytext=(8.75,2.70),arrowprops={'arrowstyle':'->','linewidth':1.4})

ax_structure.text(9.75,2.88,r'$y[n]$',fontsize=11,fontweight='bold',ha='center')

plt.subplots_adjust(left=0.03,right=0.98,top=0.85,bottom=0.05)

# ============================================================
# FIGURE 2 — FREQUENCY SAMPLES AND FREQUENCY RESPONSE
# ============================================================

fig2,(ax_samples,ax_response) = plt.subplots(1,2,figsize=(9.0,3.6))

fig2.canvas.toolbar_visible = False
fig2.canvas.header_visible = False
fig2.canvas.footer_visible = False

# ------------------------------------------------------------
# FREQUENCY SAMPLES
# ------------------------------------------------------------

sample_indices = np.arange(M)

sample_stems = ax_samples.vlines(sample_indices,0,np.abs(Hk),linewidth=1.5)

sample_markers, = ax_samples.plot(sample_indices,np.abs(Hk),'o',markersize=5)

ax_samples.axhline(0,linewidth=0.8)

ax_samples.set_xlim(-0.5,M-0.5)

ax_samples.set_ylim(0,1.10)

ax_samples.set_xticks(np.arange(M))

ax_samples.set_title('Frequency Samples |H[k]|')

ax_samples.set_xlabel('DFT bin k')

ax_samples.set_ylabel('|H[k]|')

ax_samples.grid(True,linestyle=':',alpha=0.25)

# ------------------------------------------------------------
# COMPLETE FIR FREQUENCY RESPONSE
# ------------------------------------------------------------

response_line, = ax_response.plot(omega/np.pi,np.abs(H_response),linewidth=1.4)

active_sample_1, = ax_response.plot([2*k0/M],[A],'o',markersize=6)

active_sample_2, = ax_response.plot([2*(M-k0)/M],[A],'o',markersize=6)

ax_response.set_xlim(0,2)

ax_response.set_ylim(0,1.10)

ax_response.set_title('Resulting FIR Frequency Response')

ax_response.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax_response.set_ylabel(r'$|H(e^{j\omega})|$')

ax_response.grid(True,linestyle=':',alpha=0.25)

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# FIGURE 3 — INTERNAL SIGNALS
# ============================================================

fig3,(ax_resonators,ax_comb) = plt.subplots(1,2,figsize=(9.0,3.6))

fig3.canvas.toolbar_visible = False
fig3.canvas.header_visible = False
fig3.canvas.footer_visible = False

# ------------------------------------------------------------
# RESONATOR OUTPUTS
# ------------------------------------------------------------

resonator1_line, = ax_resonators.plot(n,np.real(A*v1),linewidth=1.2,label=r'Re{$A v_{k_0}[n]$}')

resonator2_line, = ax_resonators.plot(n,np.real(A*v2),'--',linewidth=1.2,label=r'Re{$A v_{M-k_0}[n]$}')

ax_resonators.set_xlim(0,N-1)

ax_resonators.set_ylim(-INTERNAL_LIMIT,INTERNAL_LIMIT)

ax_resonators.set_title('Internal Resonator Outputs')

ax_resonators.set_xlabel('Sample index n')

ax_resonators.set_ylabel('Amplitude')

ax_resonators.grid(True,linestyle=':',alpha=0.30)

ax_resonators.legend(loc='upper right')

# ------------------------------------------------------------
# SUM BEFORE AND AFTER COMB
# ------------------------------------------------------------

sum_line, = ax_comb.plot(n,np.real(resonator_sum),linewidth=1.2,label='Resonator sum')

comb_output_line, = ax_comb.plot(n,y_fs,'--',linewidth=1.3,label='After comb section')

ax_comb.set_xlim(0,N-1)

ax_comb.set_ylim(-INTERNAL_LIMIT,INTERNAL_LIMIT)

ax_comb.set_title('Comb-Section Operation')

ax_comb.set_xlabel('Sample index n')

ax_comb.set_ylabel('Amplitude')

ax_comb.grid(True,linestyle=':',alpha=0.30)

ax_comb.legend(loc='upper right')

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# FIGURE 4 — FINAL OUTPUT AND NUMERICAL DIFFERENCE
# ============================================================

fig4,(ax_output,ax_difference) = plt.subplots(1,2,figsize=(9.0,3.5))

fig4.canvas.toolbar_visible = False
fig4.canvas.header_visible = False
fig4.canvas.footer_visible = False

# ------------------------------------------------------------
# FINAL OUTPUT COMPARISON
# ------------------------------------------------------------

direct_line, = ax_output.plot(n,y_direct,linewidth=1.5,label='Direct FIR implementation')

fs_line, = ax_output.plot(n,y_fs,'--',linewidth=1.3,label='Frequency-sampling implementation')

ax_output.set_xlim(0,N-1)

ax_output.set_ylim(-OUTPUT_LIMIT,OUTPUT_LIMIT)

ax_output.set_title('Final Output Comparison')

ax_output.set_xlabel('Sample index n')

ax_output.set_ylabel('y[n]')

ax_output.grid(True,linestyle=':',alpha=0.30)

ax_output.legend(loc='upper right')

# ------------------------------------------------------------
# NUMERICAL DIFFERENCE
# ------------------------------------------------------------

difference_line, = ax_difference.plot(n,difference,linewidth=1.2)

ax_difference.axhline(0,linewidth=0.8)

ax_difference.set_xlim(0,N-1)

ax_difference.set_ylim(-DIFFERENCE_LIMIT,DIFFERENCE_LIMIT)

ax_difference.set_title('Numerical Difference')

ax_difference.set_xlabel('Sample index n')

ax_difference.set_ylabel(r'$y_D[n]-y_{FS}[n]$')

ax_difference.grid(True,linestyle=':',alpha=0.30)

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# UPDATE CALLBACK
# ============================================================

def update(change=None):

    k0 = k_slider.value
    A = amplitude_slider.value

    # --------------------------------------------------------
    # FREQUENCY SAMPLES
    # --------------------------------------------------------

    Hk_new = create_frequency_samples(k0,A)

    # --------------------------------------------------------
    # DIRECT FIR IMPLEMENTATION
    # --------------------------------------------------------

    h_new,y_direct_new = direct_fir_output(x,Hk_new)

    omega_new,H_response_new = calculate_response(h_new)

    # --------------------------------------------------------
    # FREQUENCY-SAMPLING IMPLEMENTATION
    # --------------------------------------------------------

    v1_new,v2_new,resonator_sum_new,y_fs_new = frequency_sampling_output(x,k0,A)

    difference_new = y_direct_new-y_fs_new

    # --------------------------------------------------------
    # UPDATE STRUCTURE LABELS
    # --------------------------------------------------------

    upper_text.set_text(rf'$H[{k0}]\,/\,(1-e^{{j2\pi {k0}/{M}}}z^{{-1}})$')

    lower_text.set_text(rf'$H[{M-k0}]\,/\,(1-e^{{j2\pi {M-k0}/{M}}}z^{{-1}})$')

    # --------------------------------------------------------
    # UPDATE FREQUENCY-SAMPLE STEMS
    # --------------------------------------------------------

    sample_stems.set_segments([[(k,0),(k,np.abs(Hk_new[k]))] for k in range(M)])

    sample_markers.set_ydata(np.abs(Hk_new))

    # --------------------------------------------------------
    # UPDATE COMPLETE FREQUENCY RESPONSE
    # --------------------------------------------------------

    response_line.set_ydata(np.abs(H_response_new))

    active_sample_1.set_xdata([2*k0/M])
    active_sample_1.set_ydata([A])

    active_sample_2.set_xdata([2*(M-k0)/M])
    active_sample_2.set_ydata([A])

    # --------------------------------------------------------
    # UPDATE INTERNAL RESONATOR SIGNALS
    # --------------------------------------------------------

    resonator1_line.set_ydata(np.real(A*v1_new))

    resonator2_line.set_ydata(np.real(A*v2_new))

    sum_line.set_ydata(np.real(resonator_sum_new))

    comb_output_line.set_ydata(y_fs_new)

    # --------------------------------------------------------
    # UPDATE FINAL OUTPUTS
    # --------------------------------------------------------

    direct_line.set_ydata(y_direct_new)

    fs_line.set_ydata(y_fs_new)

    difference_line.set_ydata(difference_new)

    # --------------------------------------------------------
    # NUMERICAL INFORMATION
    # --------------------------------------------------------

    maximum_difference = np.max(np.abs(difference_new))

    omega0 = 2*np.pi*k0/M

    result_html.value = f"""
    <div class="fs-root">

    <div class="fs-box fs-result">

    <div class="fs-title">
    Current frequency-sampling implementation
    </div>

    Active frequency samples:

    <b>
    H[{k0}] = H[{M-k0}] = {A:.2f}
    </b>

    &nbsp;&nbsp;&nbsp;

    Sampling frequency:

    <b>
    ω₀ = {omega0/np.pi:.3f}π rad/sample
    </b>

    <br><br>

    FIR impulse response:

    <div class="fs-equation">
    h[n] =
    (2 × {A:.2f}/{M})
    cos(2π × {k0}n/{M}),
    &nbsp; 0 ≤ n ≤ {M-1}
    </div>

    Maximum
    |y<sub>direct</sub>[n] − y<sub>FS</sub>[n]|:

    <b>{maximum_difference:.3e}</b>

    </div>

    </div>
    """

    # --------------------------------------------------------
    # REDRAW EXISTING CANVASES
    # --------------------------------------------------------

    fig1.canvas.draw_idle()

    fig2.canvas.draw_idle()

    # Force immediate redraw of the internal-signal figure.
    fig3.canvas.draw()

    fig4.canvas.draw_idle()

# ============================================================
# OBSERVERS
# ============================================================

k_slider.observe(update,names='value')

amplitude_slider.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(result_html)

display(fig1.canvas)

display(controls)

display(fig2.canvas)

display(fig3.canvas)

display(fig4.canvas)

# ============================================================
# INITIAL UPDATE
# ============================================================

update()